# Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import pickle
import os
warnings.filterwarnings('ignore')

# Importing Train, Test and Validation datasets

In [ ]:
! gdown 1hMf9O4B4R_wDWBEcHjFuL23EgAVxSPhZ
! gdown 1xodfn2ydK-5N1tgioHIdi8d7vBEtP20c
! gdown 1dn731As0myZMyq886-XLBfoi29mYxRes
! gdown 1zmfYaZqugZ2YjnCF4MKSrIb_Dy8pmiBD
! gdown 1HtRyANvD4882oZRWtgNQHFUnbs4gjdyh
! gdown 1Az4qS0h_xHgR62O338949jrlwfMal11F

In [ ]:
X_train = joblib.load('/content/X_train.pkl')
X_test  = joblib.load('/content/X_test.pkl')
X_val   = joblib.load('/content/X_val.pkl')
y_train = joblib.load('/content/y_train.pkl')
y_test  = joblib.load('/content/y_test.pkl')
y_val   = joblib.load('/content/y_val.pkl')

print('X_train shape:', X_train.shape)
print('X_val   shape:', X_val.shape)
print('X_test  shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_val   shape:', y_val.shape)
print('y_test  shape:', y_test.shape)

# Why Logistic Regression?

**1. Problem Type**  
Toxicity classification is a binary problem (toxic = 1, non-toxic = 0).  
Logistic Regression is the canonical regression model for binary outcomes — it models the probability that a sample belongs to the positive class via the sigmoid function, which directly maps to a classification decision.

**2. Interpretability**  
The learned coefficients reveal which SVD dimensions (and by extension, which vocabulary directions) push a comment towards toxicity. This interpretability is valuable for understanding model decisions in a content-moderation setting.

**3. Strong Baseline on TF-IDF Features**  
Logistic Regression is well-known to perform strongly on high-dimensional sparse text features (and their SVD-reduced dense counterparts). It is a natural complement to the Naïve Bayes baseline already implemented by our team.

**4. Regularisation Options**  
Scikit-learn exposes L1 (Lasso), L2 (Ridge), and ElasticNet penalties, allowing us to control overfitting and perform implicit feature selection — important given 300 SVD dimensions.

**5. Calibrated Probabilities & ROC-AUC**  
Logistic Regression outputs well-calibrated probabilities, making the ROC-AUC metric especially meaningful and enabling future threshold adjustment for precision/recall trade-offs.

# Model Assumptions

| Assumption | How We Address It |
|---|---|
| Linear decision boundary in feature space | TF-IDF + TruncatedSVD maps raw text into a 300-dimensional dense space where linear separability is reasonable |
| Features are numerical | X_train/X_val/X_test are already float64 SVD-reduced matrices — no conversion needed |
| Features are not perfectly multicollinear | TruncatedSVD produces orthogonal components, eliminating multicollinearity by construction |
| Observations are independent | Each tweet/comment is independently sampled from the dataset |
| No extreme outliers distorting the decision boundary | StandardScaler (applied during Milestone 2 preprocessing) normalises each dimension |
| Large enough sample | Training set has ~47K samples — well above the rule-of-thumb minimum for stable logistic regression |

# Hyperparameter Tuning

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Base estimator — solver='saga' supports l1, l2, and elasticnet penalties
lr = LogisticRegression(max_iter=1000, random_state=42, solver='saga')

param_grid = {
    'C':       [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(
    estimator  = lr,
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'f1',
    n_jobs     = -1,
    verbose    = 1
)

print('Parameter grid:')
print(param_grid)

In [ ]:
grid_lr.fit(X_train, y_train)

print('Best Params :', grid_lr.best_params_)
print('Best CV F1  :', round(grid_lr.best_score_, 4))

In [ ]:
# Visualise cross-validation F1 scores across the grid
cv_results = pd.DataFrame(grid_lr.cv_results_)
pivot = cv_results.pivot_table(
    index   = 'param_C',
    columns = 'param_penalty',
    values  = 'mean_test_score'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='Blues', linewidths=0.5)
plt.title('GridSearchCV — Mean CV F1 Score (Logistic Regression)', fontsize=13, fontweight='bold')
plt.xlabel('Penalty')
plt.ylabel('C  (inverse regularisation strength)')
plt.tight_layout()
plt.show()

# Training Logistic Regression

In [ ]:
final_model = grid_lr.best_estimator_
print('Final model:', final_model)

y_val_pred  = final_model.predict(X_val)
y_pred      = final_model.predict(X_test)
y_pred_prob = final_model.predict_proba(X_test)[:, 1]  # probability of toxic class

print('\nSample predictions (first 20):', y_pred[:20])

# Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# --- Validation set ---
print('===== Validation Set =====')
print('Accuracy :', round(accuracy_score (y_val, y_val_pred), 4))
print('Precision:', round(precision_score(y_val, y_val_pred), 4))
print('Recall   :', round(recall_score   (y_val, y_val_pred), 4))
print('F1 Score :', round(f1_score       (y_val, y_val_pred), 4))

print()

# --- Test set ---
print('===== Test Set =====')
print('Accuracy  of Logistic Regression:', round(accuracy_score (y_test, y_pred),      4))
print('Precision of Logistic Regression:', round(precision_score(y_test, y_pred),      4))
print('Recall    of Logistic Regression:', round(recall_score   (y_test, y_pred),      4))
print('F1 Score  of Logistic Regression:', round(f1_score       (y_test, y_pred),      4))
print('ROC-AUC   of Logistic Regression:', round(roc_auc_score  (y_test, y_pred_prob), 4))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

sns.reset_orig()
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['Non-Toxic (0)', 'Toxic (1)']))

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#1e3c72', lw=2,
         label=f'Logistic Regression (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='grey', lw=1.5, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Logistic Regression', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 most influential features (SVD dimensions) for toxicity
coef = final_model.coef_[0]
n_features = min(20, len(coef))

top_pos_idx = np.argsort(coef)[-n_features:][::-1]
top_neg_idx = np.argsort(coef)[:n_features]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 20 Most Influential SVD Features', fontsize=14, fontweight='bold')

axes[0].barh([f'SVD_{i}' for i in top_pos_idx], coef[top_pos_idx],
             color='#c31432', edgecolor='white')
axes[0].set_title('Top 20 — Towards Toxic (Positive Coefficient)')
axes[0].set_xlabel('Coefficient Value')
axes[0].invert_yaxis()

axes[1].barh([f'SVD_{i}' for i in top_neg_idx], coef[top_neg_idx],
             color='#1e3c72', edgecolor='white')
axes[1].set_title('Top 20 — Towards Non-Toxic (Negative Coefficient)')
axes[1].set_xlabel('Coefficient Value')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Challenges Faced & Solutions

| Challenge | Solution |
|---|---|
| Choosing the right penalty (L1 vs L2) | Used GridSearchCV over both; L2 tends to win on dense SVD features but L1 was included to allow sparse coefficient selection |
| `max_iter` convergence warnings | Set `max_iter=1000` and used `solver='saga'` which converges faster on large datasets |
| Class imbalance in raw dataset | Preprocessed pkl files already account for stratified splits from Milestone 2; additionally `class_weight` can be set to `'balanced'` if needed |
| High-dimensional feature space (300 dims) | Regularisation parameter C controls the bias-variance trade-off; smaller C = stronger regularisation |
| Grid search run time | `n_jobs=-1` parallelises across all available CPU cores |

# Before and After Data Transformation Snapshots

In [ ]:
# BEFORE — raw text (original combined_model_dataset.csv)
# Loaded here for illustration; not used for model training
! gdown 11uurHsNZAjG70Ooq0noI7a2W4rbiNvIw  # combined_model_dataset.csv

df_raw = pd.read_csv('/content/combined_model_dataset.csv')
print('BEFORE — Raw Text:')
print(df_raw[['text', 'label']].head(5).to_string(index=False))
print('\nShape :', df_raw.shape)
print('Dtype :', df_raw['text'].dtype)

In [ ]:
# AFTER — TF-IDF → TruncatedSVD → StandardScaler (loaded as X_train)
import scipy.sparse

X_train_dense = X_train.toarray() if scipy.sparse.issparse(X_train) else np.array(X_train)

df_after = pd.DataFrame(
    X_train_dense[:5],
    columns=[f'SVD_{i}' for i in range(X_train_dense.shape[1])]
)

print('AFTER — TF-IDF → TruncatedSVD → StandardScaler:')
print(df_after.iloc[:, :10].to_string())   # show first 10 columns for readability
print(f'\nFull shape: {X_train_dense.shape}')
print(f'Dtype     : {X_train_dense.dtype}')

In [ ]:
# Summary: transformation pipeline
summary = pd.DataFrame({
    'Stage': [
        'Raw Text',
        'After TF-IDF Vectorisation',
        'After TruncatedSVD (300 components)',
        'After StandardScaler'
    ],
    'Format': [
        'String',
        'Sparse float64 matrix',
        'Dense float64 matrix',
        'Dense float64 matrix (zero mean, unit variance)'
    ],
    'Typical Shape': [
        '(N,)',
        '(N, ~100K vocab)',
        '(N, 300)',
        '(N, 300)'
    ],
    'Usable by LR?': ['No', 'Yes (sparse)', 'Yes (dense)', 'Yes (optimal)']
})
print(summary.to_string(index=False))

# Saving the Model

In [ ]:
model_path = '/content/logistic_regression_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)

print('Model saved at :', model_path)
print('File exists    :', os.path.exists(model_path))

In [ ]:
from google.colab import files
files.download('/content/logistic_regression_model.pkl')